In [0]:
%sql
create or replace temp view raw_unstructured as
select path, content
from read_files("/Volumes/workspace/default/vol_ai")

In [0]:
%sql
select path,
      length(content)
  from raw_unstructured

In [0]:
%sql
create or replace temp view parsed_structured_doc as
select path ,
ai_parse_document(content) as parsed_content
from raw_unstructured

In [0]:
%sql
select path ,parsed_content
from parsed_structured_doc

In [0]:
%sql
create or replace temp view parsed_classified_doc as
select *,
ai_classify(cast(parsed_content  as String), array('Invoice','Admin','Receipt','Purchase order')) as label
from parsed_structured_doc

In [0]:
%sql
create or replace temp view structured_tables as 
select path, label,
try_cast(e:content as string) as table_html
from parsed_classified_doc
lateral view explode(try_cast(parsed_content:document:elements as array<VARIANT>)) t as e
where try_cast(e:type as string ) = 'table'
    


In [0]:
%sql
select * from structured_tables

In [0]:
%sql
select label, 
       ai_extract(table_html, array('CPT Code')).`CPT Code` as CPT_Code,
       ai_extract(table_html, array('ICD Code')).`ICD Code` as ICD_Code,
       ai_extract(table_html, array('Description')).`Description` as Description,
       ai_extract(table_html, array('Billed Amount')).`Billed Amount` as Billed_Amount,
       ai_extract(table_html, array('Paid Amount')).`Paid Amount` as Paid_Amount
from structured_tables


In [0]:
%sql
WITH rows AS (
    SELECT *,
           explode(regexp_extract_all(table_html,'<tr>(.*?)</tr>',1)) AS tr
    FROM structured_tables
),
data_rows AS (
    SELECT path,Label,tr
    FROM rows
    WHERE tr LIKE '%<td>%'
)
select label,
       ai_extract(tr, array('CPT Code')).`CPT Code` as CPT_Code,
       ai_extract(tr, array('ICD Code')).`ICD Code` as ICD_Code,
       ai_extract(tr, array('Description')).`Description` as Description,
       ai_extract(tr, array('Billed Amount')).`Billed Amount` as Billed_Amount,
       ai_extract(tr, array('Paid Amount')).`Paid Amount` as Paid_Amount
from data_rows